In [1]:
import sys
print(sys.executable)

c:\Coding\sem5\ml\projects\classification\titanic-survival-prediction\.venv\Scripts\python.exe


In [2]:
import pandas as pd
print(pd.__version__)

3.0.6


In [3]:
df = pd.read_csv("../data/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.shape

(891, 12)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
missing = df.isnull().sum()
missing_percentage = (missing/len(df))*100
pd.DataFrame({
    "Missing Values":missing,
    "Missing Percentage": missing_percentage
})

,Missing Values,Missing Percentage
PassengerId,0,0.000000
Survived,0,0.000000
Pclass,0,0.000000
Name,0,0.000000
Sex,0,0.000000
Age,177,19.865320
SibSp,0,0.000000
Parch,0,0.000000
Ticket,0,0.000000
Fare,0,0.000000


In [9]:
df["Survived"].value_counts()
df["Survived"].value_counts(normalize=True) * 100

Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64

In [10]:
categorical_columns = ["Sex", "Embarked", "Pclass"]
for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- Sex ---
Sex
male      577
female    314
Name: count, dtype: int64

--- Embarked ---
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

--- Pclass ---
Pclass
3    491
1    216
2    184
Name: count, dtype: int64


In [11]:
for column in categorical_columns:
    print(f"\n--- Survival by {column} ---")
    print(
        pd.crosstab(
            df[column],
            df["Survived"],
            normalize="index"
        ) * 100
    )


--- Survival by Sex ---
Survived          0          1
Sex                           
female    25.796178  74.203822
male      81.109185  18.890815

--- Survival by Embarked ---
Survived          0          1
Embarked                      
C         44.642857  55.357143
Q         61.038961  38.961039
S         66.304348  33.695652

--- Survival by Pclass ---
Survived          0          1
Pclass                        
1         37.037037  62.962963
2         52.717391  47.282609
3         75.763747  24.236253


In [12]:
numerical_columns = ["Age", "Fare", "SibSp", "Parch"]

df[numerical_columns].describe()

,Age,Fare,SibSp,Parch
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008,0.381594
std,14.526497,49.693429,1.102743,0.806057
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,7.910400,0.000000,0.000000
50%,28.000000,14.454200,0.000000,0.000000
75%,38.000000,31.000000,1.000000,0.000000
max,80.000000,512.329200,8.000000,6.000000


In [13]:
df.groupby("Survived")["Age"].agg(["count", "mean", "median", "min", "max"])

,count,mean,median,min,max
Survived,,,,,
0,424,30.626179,28.0,1.00,74.0
1,290,28.343690,28.0,0.42,80.0


In [14]:
for column in numerical_columns:
    print(f"\n--- {column} ---")
    print(
        df.groupby("Survived")[column].agg(
            ["mean","median", "min", "max"]
        )
    )


--- Age ---
               mean  median   min   max
Survived                               
0         30.626179    28.0  1.00  74.0
1         28.343690    28.0  0.42  80.0

--- Fare ---
               mean  median  min       max
Survived                                  
0         22.117887    10.5  0.0  263.0000
1         48.395408    26.0  0.0  512.3292

--- SibSp ---
              mean  median  min  max
Survived                            
0         0.553734     0.0    0    8
1         0.473684     0.0    0    4

--- Parch ---
              mean  median  min  max
Survived                            
0         0.329690     0.0    0    6
1         0.464912     0.0    0    5


In [15]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df.groupby("Survived")["FamilySize"].agg(
    ["mean", "median", "min", "max"]
)

,mean,median,min,max
Survived,,,,
0,1.883424,1.0,1,11
1,1.938596,2.0,1,7


In [17]:
family_survival = pd.crosstab(
    df["FamilySize"],
    df["Survived"],
    normalize = "index"
) * 100

family_survival

Survived,0,1
FamilySize,,
1,69.646182,30.353818
2,44.720497,55.279503
3,42.156863,57.843137
4,27.586207,72.413793
5,80.000000,20.000000
6,86.363636,13.636364
7,66.666667,33.333333
8,100.000000,0.000000
11,100.000000,0.000000


In [18]:
df[["Name", "Survived"]].head(10)

,Name,Survived
0,"Braund, Mr. Owen Harris",0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1
2,"Heikkinen, Miss. Laina",1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1
4,"Allen, Mr. William Henry",0
5,"Moran, Mr. James",0
6,"McCarthy, Mr. Timothy J",0
7,"Palsson, Master. Gosta Leonard",0
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",1
9,"Nasser, Mrs. Nicholas (Adele Achem)",1


In [19]:
df["Name"].head(10).tolist()

['Braund, Mr. Owen Harris',
 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
 'Heikkinen, Miss. Laina',
 'Futrelle, Mrs. Jacques Heath (Lily May Peel)',
 'Allen, Mr. William Henry',
 'Moran, Mr. James',
 'McCarthy, Mr. Timothy J',
 'Palsson, Master. Gosta Leonard',
 'Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)',
 'Nasser, Mrs. Nicholas (Adele Achem)']

In [20]:
df["Title"] = df["Name"].str.extract(r", ([^.]*)\.")

In [21]:
df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [22]:
title_survival = pd.crosstab(
    df["Title"],
    df["Survived"],
    normalize="index"
) * 100

title_survival

Survived,0,1
Title,,
Capt,100.000000,0.000000
Col,50.000000,50.000000
Don,100.000000,0.000000
Dr,57.142857,42.857143
Jonkheer,100.000000,0.000000
Lady,0.000000,100.000000
Major,50.000000,50.000000
Master,42.500000,57.500000
Miss,30.219780,69.780220


In [23]:
df[["Cabin", "Survived"]].head(20)

,Cabin,Survived
0,NaN,0
1,C85,1
2,NaN,1
3,C123,1
4,NaN,0
5,NaN,0
6,E46,0
7,NaN,0
8,NaN,1
9,NaN,1


In [24]:
df["Cabin"].notna().sum(), df["Cabin"].isna().sum()

(np.int64(204), np.int64(687))

In [25]:
df["CabinKnown"] = df["Cabin"].notna()

pd.crosstab(
    df["CabinKnown"],
    df["Survived"],
    normalize="index"
) * 100

Survived,0,1
CabinKnown,,
False,70.014556,29.985444
True,33.333333,66.666667


In [26]:
pd.crosstab(
    df["CabinKnown"],
    df["Pclass"],
    normalize="index"
) * 100

Pclass,1,2,3
CabinKnown,,,
False,5.822416,24.454148,69.723435
True,86.274510,7.843137,5.882353


In [27]:
df[["Ticket", "Survived"]].head(20)

,Ticket,Survived
0,A/5 21171,0
1,PC 17599,1
2,STON/O2. 3101282,1
3,113803,1
4,373450,0
5,330877,0
6,17463,0
7,349909,0
8,347742,1
9,237736,1


In [28]:
df["Ticket"].nunique(), df["Ticket"].value_counts().head(15)

(681,
 Ticket
 347082          7
 1601            7
 CA. 2343        7
 3101295         6
 CA 2144         6
 347088          6
 382652          5
 S.O.C. 14879    5
 349909          4
 347077          4
 19950           4
 W./C. 6608      4
 4133            4
 LINE            4
 113781          4
 Name: count, dtype: int64)

In [29]:
df["TicketGroupSize"] = df.groupby("Ticket")["Ticket"].transform("count")

df["TicketGroupSize"].describe()

count    891.000000
mean       1.787879
std        1.361142
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        7.000000
Name: TicketGroupSize, dtype: float64

In [30]:
pd.crosstab(
    df["TicketGroupSize"],
    df["Survived"],
    normalize="index"
) * 100

Survived,0,1
TicketGroupSize,,
1,70.201097,29.798903
2,42.553191,57.446809
3,30.158730,69.841270
4,50.000000,50.000000
5,100.000000,0.000000
6,100.000000,0.000000
7,76.190476,23.809524


In [31]:
df.groupby(["Sex", "Survived"])["Age"].agg(
    ["count", "mean", "median"]
)

count       mean  median
Sex    Survived                          
female 0            64  25.046875    24.5
       1           197  28.847716    28.0
male   0           360  31.618056    29.0
       1            93  27.276022    28.0

In [32]:
df.groupby(["Pclass", "Survived"])["Age"].agg(
    ["count", "mean", "median"]
)

count       mean  median
Pclass Survived                          
1      0            64  43.695312   45.25
       1           122  35.368197   35.00
2      0            90  33.544444   30.50
       1            83  25.901566   28.00
3      0           270  26.555556   25.00
       1            85  20.646118   22.00

## 5. Preprocessing

In [34]:
# Separate features and target
X = df.drop(columns=["Survived"])
y = df["Survived"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (891, 15)
Target shape: (891,)


In [35]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify = y
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)

Training set: (712, 15)
Validation set: (179, 15)


In [36]:
X.columns.tolist()

['PassengerId',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked',
 'FamilySize',
 'Title',
 'CabinKnown',
 'TicketGroupSize']

## 5.1 Preprocessing Pipeline

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "CabinKnown",
    "TicketGroupSize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)

Processed training shape: (712, 27)
Processed validation shape: (179, 27)


In [41]:
feature_names = preprocessor.get_feature_names_out()
print("Number of processed features: ", len(feature_names))
print(feature_names)

Number of processed features:  27
['num__Pclass' 'num__Age' 'num__SibSp' 'num__Parch' 'num__Fare'
 'num__FamilySize' 'num__CabinKnown' 'num__TicketGroupSize'
 'cat__Sex_female' 'cat__Sex_male' 'cat__Embarked_C' 'cat__Embarked_Q'
 'cat__Embarked_S' 'cat__Title_Col' 'cat__Title_Don' 'cat__Title_Dr'
 'cat__Title_Jonkheer' 'cat__Title_Lady' 'cat__Title_Major'
 'cat__Title_Master' 'cat__Title_Miss' 'cat__Title_Mlle' 'cat__Title_Mr'
 'cat__Title_Mrs' 'cat__Title_Ms' 'cat__Title_Rev' 'cat__Title_Sir']


In [42]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train_processed, y_train)

y_val_pred = baseline.predict(X_val_processed)

print("Baseline Accuracy: ", accuracy_score(y_val, y_val_pred))
print()
print(classification_report(y_val, y_val_pred))

Baseline Accuracy:  0.6145251396648045

              precision    recall  f1-score   support

           0       0.61      1.00      0.76       110
           1       0.00      0.00      0.00        69

    accuracy                           0.61       179
   macro avg       0.31      0.50      0.38       179
weighted avg       0.38      0.61      0.47       179



c:\Coding\sem5\ml\projects\classification\titanic-survival-prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Coding\sem5\ml\projects\classification\titanic-survival-prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Coding\sem5\ml\projects\classification\titanic-survival-prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zer

In [43]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

y_val_pred = logistic_model.predict(X_val_processed)

print("Logistic Regression Accuracy:", accuracy_score(y_val, y_val_pred))
print()
print(classification_report(y_val, y_val_pred))

Logistic Regression Accuracy: 0.8379888268156425

              precision    recall  f1-score   support

           0       0.86      0.88      0.87       110
           1       0.80      0.77      0.79        69

    accuracy                           0.84       179
   macro avg       0.83      0.82      0.83       179
weighted avg       0.84      0.84      0.84       179

